# 3D-VGA — train and evaluate one model

Trains one audio grounding model and scores it. Built for **Run All**.

Only the acoustic branch and the fusion head are trained. Both inputs are read from disk — detector proposals from `cached_scenes/`, speech features from `data_audio/feat/` — so no detector and no speech encoder runs.

| step | what it does | needs |
|---|---|---|
| **check** | model self-test on synthetic tensors | seconds, any machine |
| **train** | one arm of `experiments/audio/fusion_arms.py` | GPU |
| **evaluate** | Unique / Multiple / Overall at IoU 0.25 and 0.5 | GPU, minutes |

Before this: `run_audio_encode.ipynb`. After this: `run_audio_analysis.ipynb` for the ablations.

## 1 — configuration

The only cell you need to edit.

In [ ]:
# USER CONFIGURATION

DEVICE     = "auto"      # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID     = "0"
FORCE_CPU  = False       # train without a usable GPU (very slow; for a quick test only)
QUICK_TEST = False       # True: 2 scenes, 2 epochs, tagged QUICK-, reads data_audio/_quick
                         #       (build that corpus with QUICK_TEST in run_audio_encode.ipynb)

ARM = "spot"             # which arm to train: "spot" is the proposed model.
                         # `python experiments/audio/fusion_arms.py list` shows the others.

USE_CACHED_SCENES  = True
CACHED_SCENES_ROOT = "cached_scenes"
ENCODER            = "wav2vec2"     # which feature cache to read

EPOCH        = 30
BATCH_SIZE   = 8         # scenes per batch
LANG_NUM_MAX = 8         # spoken descriptions per scene per batch
LR           = 1e-3
SEED         = 42
VAL_STEP     = 2000      # validate every N ITERATIONS; a checkpoint is saved only then
VERBOSE      = 50        # print a training line every N iterations
NUM_WORKERS  = 4
EXTRA_ARGS   = ["--use_color", "--use_normal", "--coslr"]   # MUST match the scene cache

# Checkpoint that evaluation scores. model_criteria_25.pth is the best validation
# Acc@0.25 seen during training; model_last.pth is the final epoch.
CHECKPOINT_FILE = "model_criteria_25.pth"

RUN_SELFTEST  = True     # model wiring on synthetic tensors, ~10 s
RUN_SMOKETEST = False    # the whole pipeline on a 40-utterance corpus, ~5 min (needs piper-tts)
RUN_TRAIN     = True
RUN_EVAL      = True

# GOOGLE DRIVE -- Colab only
MIRROR_TO_DRIVE = True
DRIVE_FOLDER    = "3DVGA"           # under MyDrive/

QUICK_SCENES = 2

## 2 — environment

Probes the GPU by launching a kernel — `torch.cuda.is_available()` alone can report `True` on an unsupported card.

In [ ]:
import glob as _glob
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "audio_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

# lib/config.py lists this directory at import time; an empty one is enough for the steps
# that never read a scan.
(REPO / "data" / "scannet" / "scans").mkdir(parents=True, exist_ok=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"quick test   : {QUICK_TEST}")
print("=" * 88)

## 3 — packages

Colab only. The audio model needs no `pyg-lib`: its spatial graph is a dense kNN.

In [ ]:
EXTRA_PACKAGES = ["h5py"]
REQUIRED_MODULES = ["torch", "numpy", "h5py", "easydict", "plyfile", "tensorboardX", "tqdm"]

def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    if (REPO / "req" / "requirements.txt").is_file():
        _pip("-r", str(REPO / "req" / "requirements.txt"))
    _pip(*EXTRA_PACKAGES)

ensure_packages()

print("\n-- packages --")
MISSING_PACKAGES = []
for module in REQUIRED_MODULES:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        MISSING_PACKAGES.append(module)
        print(f"  {module:18s} --    MISSING")
if MISSING_PACKAGES:
    print(f"\n  install first: pip install {' '.join(MISSING_PACKAGES)}")

## 4 — Google Drive

Colab runtimes are disposable — `outputs/` is copied to Drive after every step, and the feature caches and earlier runs are brought back on a fresh runtime.

In [ ]:
# Off Colab this whole cell is inert, so the notebook stays runnable locally.

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, everything stays local"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as error:
        # A failed mount must not take the run with it.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def copy_newer(source, target, quiet=False):
    """Copy a file or a tree, skipping files the target already has up to date."""
    source, target = Path(source), Path(target)
    if not source.exists():
        return 0
    files = [source] if source.is_file() else [p for p in source.rglob("*") if p.is_file()]
    copied = 0
    for path in files:
        if "tensorboard" in path.parts or path.suffix == ".tmp":
            continue
        dest = target if source.is_file() else target / path.relative_to(source)
        try:
            if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime \
                    and dest.stat().st_size == path.stat().st_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    return copied

def mirror_to_drive(quiet=False):
    """Copy outputs/ to Drive. Cheap to call repeatedly: only newer files are copied."""
    if DRIVE_ROOT is None:
        return 0
    copied = copy_newer(REPO / "outputs", DRIVE_ROOT / "outputs", quiet=quiet)
    if not quiet:
        print(f"mirrored {copied} new/changed file(s) to {DRIVE_ROOT}/outputs")
    return copied

def restore_from_drive(*relative):
    """Bring files back from Drive after a fresh runtime, where they are missing here."""
    if DRIVE_ROOT is None:
        return
    for rel in relative:
        copied = copy_newer(DRIVE_ROOT / rel, REPO / rel, quiet=True)
        if copied:
            print(f"restored {copied} file(s) from Drive: {rel}")

mount_drive()

if not QUICK_TEST:
    restore_from_drive("data_audio/feat", "data_audio/manifest_train.json",
                       "data_audio/manifest_val.json", "outputs")

### Runner

Streams a command's output and keeps its log under `outputs/reports/<step>/`.

In [ ]:
REPORTS = REPO / "outputs" / "reports"

def run(key, cmd, env=None, note=""):
    """Run one command, stream its output, and keep a report under outputs/reports/."""
    cmd = [str(c) for c in cmd]
    printable = " ".join(cmd)
    print("=" * 88)
    print(f"STEP   {key}" + (f"\n       {note}" if note else ""))
    print(f"$ {printable}\n{'-' * 88}")

    full_env = os.environ.copy()
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    full_env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    full_env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=full_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code = process.wait()
    elapsed = round(time.time() - started, 1)

    folder = REPORTS / key
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "log.txt").write_text("".join(captured))
    record = {"key": key, "note": note, "cmd": printable, "exit_code": code,
              "elapsed_sec": elapsed, "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
              "environment": "Google Colab" if IN_COLAB else "local", "gpu": GPU_NAME or "none"}
    (folder / "result.json").write_text(json.dumps(record, indent=2))
    RESULTS[key] = record

    if code in (247, -9, 137):
        verdict = f"FAILED (exit {code}) -- killed by the OS, out of host RAM"
    else:
        verdict = "OK" if code == 0 else f"FAILED (exit {code})"
    print(f"{'-' * 88}\n[{key}] {verdict} in {elapsed}s   log -> {folder.relative_to(REPO)}/log.txt")
    return code

## 5 — apply the configuration

The runner scripts read their shared settings from `VGA_<NAME>` environment variables, exported here — no source file is rewritten. Re-run this cell after editing the config cell.

In [ ]:
USE_CPU = (DEVICE == "cpu") or not GPU_USABLE
AUDIO_ROOT = REPO / "data_audio" / ("_quick" if QUICK_TEST else "")
RUN_PREFIX = "QUICK-" if QUICK_TEST else ""

TRAINING_CONFIG = {
    "USE_CACHED_SCENES":  USE_CACHED_SCENES,
    "CACHED_SCENES_ROOT": CACHED_SCENES_ROOT,
    "ENCODER":            ENCODER,
    "EPOCH":              2 if QUICK_TEST else EPOCH,
    "BATCH_SIZE":         2 if QUICK_TEST else BATCH_SIZE,
    "LANG_NUM_MAX":       LANG_NUM_MAX,
    "LR":                 LR,
    "SEED":               SEED,
    "VAL_STEP":           5 if QUICK_TEST else VAL_STEP,
    "VERBOSE":            5 if QUICK_TEST else VERBOSE,
    "NUM_WORKERS":        0 if QUICK_TEST else NUM_WORKERS,
    "NUM_SCENES":         QUICK_SCENES if QUICK_TEST else -1,
    "NUM_VAL_SCENES":     QUICK_SCENES if QUICK_TEST else -1,
    "ONLY_CACHED_AUDIO":  QUICK_TEST,
    "EXTRA_ARGS":         EXTRA_ARGS,
    "CHECKPOINT":         CHECKPOINT_FILE,
    "RUN_PREFIX":         RUN_PREFIX,
    "CPU":                USE_CPU,
    "AUDIO_ROOT":         AUDIO_ROOT,
}

def as_env(value):
    """fusion_arms._env parses these back: lists comma-separated, bools as true/false."""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (list, tuple)):
        return ",".join(str(v) for v in value)
    return str(value)

for name, value in TRAINING_CONFIG.items():
    os.environ[f"VGA_{name}"] = as_env(value)
    print(f"  VGA_{name:<20} = {as_env(value)}")

ARMS_PY = "experiments/audio/fusion_arms.py"
ANALYSIS_PY = "experiments/audio/analysis.py"
RUN_TAG = f"{RUN_PREFIX}VGA-{ARM}".upper()

def latest_run():
    runs = sorted(_glob.glob(str(REPO / "outputs" / f"*_{RUN_TAG}")))
    return Path(runs[-1]) if runs else None

## 6 — preflight

Checks the two caches the run reads and prints the command it will launch. Changes nothing.

In [ ]:
import math

problems = []

meta_path = REPO / CACHED_SCENES_ROOT / "meta.json"
if USE_CACHED_SCENES:
    if not meta_path.is_file():
        problems.append(f"no scene cache at {CACHED_SCENES_ROOT}/ (unzip cached_scenes.zip)")
    else:
        meta = json.loads(meta_path.read_text())
        print(f"scene cache   : detector {meta.get('detector')}, {meta.get('num_proposals')} "
              f"proposals, complete={meta.get('complete')}")
        if not meta.get("complete"):
            problems.append("the scene cache is marked incomplete")

def split_annotations(split):
    data = json.loads((REPO / "data" / f"ScanRefer_filtered_{split}.json").read_text())
    if QUICK_TEST:
        scenes = set(sorted({d["scene_id"] for d in data})[:QUICK_SCENES])
        data = [d for d in data if d["scene_id"] in scenes]
    return data

GROUPS = {}
for split in ("train", "val"):
    data = split_annotations(split)
    cache = AUDIO_ROOT / "feat" / ENCODER / f"{split}.h5"
    scenes = sorted({d["scene_id"] for d in data})
    if USE_CACHED_SCENES:
        absent = [s for s in scenes if not (REPO / CACHED_SCENES_ROOT / split / f"{s}.p").is_file()]
        if absent:
            problems.append(f"{len(absent)} {split} scene(s) missing from the scene cache")
    lost = [s for s in scenes
            if not (REPO / "data" / "scannet" / "scannet_data" / f"{s}_aligned_vert.npy").is_file()]
    if lost:
        problems.append(f"{len(lost)} {split} scene(s) missing from data/scannet/scannet_data "
                        f"(the ground-truth boxes are read from there)")
    if not cache.is_file():
        problems.append(f"no feature cache {cache.relative_to(REPO)} (run_audio_encode.ipynb)")
        continue
    import h5py
    with h5py.File(cache, "r") as handle:
        keys = {k.decode("utf-8") for k in handle["keys"][:]}
    have_audio = [d for d in data if f"{d['scene_id']}/{d['object_id']}/{d['ann_id']}" in keys]
    per_scene = {}
    for d in (have_audio if QUICK_TEST else data):
        per_scene[d["scene_id"]] = per_scene.get(d["scene_id"], 0) + 1
    GROUPS[split] = sum(math.ceil(n / LANG_NUM_MAX) for n in per_scene.values())
    print(f"{split:<5} audio   : {len(have_audio)} of {len(data)} description(s) have features")
    if len(have_audio) < len(data) and not QUICK_TEST:
        problems.append(f"{len(data) - len(have_audio)} {split} description(s) have no "
                        f"features and would be heard as silence")

if "train" in GROUPS:
    batch = TRAINING_CONFIG["BATCH_SIZE"]
    per_epoch = math.ceil(GROUPS["train"] / batch)
    total = per_epoch * TRAINING_CONFIG["EPOCH"]
    print(f"\ntraining      : {per_epoch} iterations/epoch x {TRAINING_CONFIG['EPOCH']} epochs "
          f"= {total} iterations, validating every {TRAINING_CONFIG['VAL_STEP']}")
    if total <= TRAINING_CONFIG["VAL_STEP"]:
        problems.append("VAL_STEP is larger than the whole run, so no checkpoint would be "
                        "saved -- lower VAL_STEP")

if USE_CPU and not (FORCE_CPU or QUICK_TEST):
    problems.append("no usable GPU (set FORCE_CPU=True to train on CPU anyway)")

print()
subprocess.run([sys.executable, ARMS_PY, "run", ARM, "--dry_run"], cwd=str(REPO))

READY = not problems
print("\n" + ("READY" if READY else "NOT READY:\n  - " + "\n  - ".join(problems)))

## 7 — is everything wired?

`selftest` builds the model on synthetic tensors and checks shapes, both transport plans, eval determinism and that gradients reach every trainable part. `smoketest` (off by default) runs the whole pipeline — speech synthesis to analysis tables — on forty utterances.

In [ ]:
# switches: RUN_SELFTEST, RUN_SMOKETEST
if RUN_SELFTEST:
    run("audio-selftest", [sys.executable, "scripts/audio/selftest.py", "--all_arms",
                           "--device", "cpu" if USE_CPU else "cuda"],
        note="every arm builds and backpropagates")
if RUN_SMOKETEST:
    run("audio-smoketest", [sys.executable, "scripts/audio/smoketest.py",
                            *(["--cpu"] if USE_CPU else [])],
        note="end-to-end on a 40-utterance corpus")

## 8 — train

Writes `outputs/<timestamp>_VGA-<ARM>/`. Watch three numbers in the validation lines: `slot_mass_max` (one slot taking everything means the slots collapsed), `role_peak` (near `1/NUM_SLOTS` means the referent role never settled on a slot) and `ot_peak` (near uniform means the transport plan carries no decision).

In [ ]:
# switch: RUN_TRAIN
if not RUN_TRAIN:
    print("RUN_TRAIN is off")
elif not READY:
    print("skipped -- preflight is NOT READY:\n  - " + "\n  - ".join(problems))
else:
    run(f"audio-train-{ARM}", [sys.executable, ARMS_PY, "run", ARM],
        note=f"train the '{ARM}' arm on cached scenes + cached audio")
    mirror_to_drive()

## 9 — evaluate

Scores the newest run of this arm over val and writes `scores.p`, `predictions.p` and `diagnostics.p` into its folder — the files every analysis in `run_audio_analysis.ipynb` reads.

In [ ]:
# switch: RUN_EVAL
if not RUN_EVAL:
    print("RUN_EVAL is off")
elif latest_run() is None:
    print(f"no run tagged {RUN_TAG} under outputs/ -- train first")
else:
    run(f"audio-eval-{ARM}", [sys.executable, ARMS_PY, "eval", ARM, "--force"],
        note=f"score {latest_run().name}")
    mirror_to_drive()

## 10 — results

The metric table, the training curves, and two health checks: whether the slots took distinct roles, and whether the errors are mishearing the object class or picking the wrong instance.

In [ ]:
import pickle

folder = latest_run()
if folder is None or not (folder / "scores.p").is_file():
    print("nothing evaluated yet")
else:
    report = pickle.load(open(folder / "scores.p", "rb"))["report"]
    print(f"run: {folder.name}\n")
    print(f"{'':10s} {'Acc@0.25':>10s} {'Acc@0.5':>10s}")
    for name in ("unique", "multiple", "overall"):
        print(f"{name.capitalize():10s} {report[name][0]:10.2f} {report[name][1]:10.2f}")
    print(f"\naudio class accuracy: {100 * report['audio_cls_acc']:.2f}%")

    log_path = folder / "my_log.json"
    if log_path.is_file():
        import matplotlib.pyplot as plt
        log = json.loads(log_path.read_text())
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
        axes[0].plot([100 * v for v in log["val"]["25"]], marker="o", label="val Acc@0.25")
        axes[0].plot([100 * v for v in log["val"]["5"]], marker="o", label="val Acc@0.5")
        axes[0].set_xlabel(f"validation pass (every {TRAINING_CONFIG['VAL_STEP']} iterations)")
        axes[0].set_ylabel("%"); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot([sum(v) / len(v) for v in log["val_loss"] if v], marker="o")
        axes[1].set_xlabel("validation pass"); axes[1].set_ylabel("val loss")
        axes[1].grid(alpha=0.3)
        plt.tight_layout(); plt.show()

    run(f"audio-slot-roles-{ARM}", [sys.executable, ANALYSIS_PY, "slot-roles",
                                    "--folder", folder.name, "--by_outcome"])
    run(f"audio-error-taxonomy-{ARM}", [sys.executable, ANALYSIS_PY, "error-taxonomy",
                                        "--folder", folder.name])